Manipulação de Dados com Python e SQL [T7] - M1S10 - Mini Projeto Módulo 1 - 21/10/2026

Aluna: Isabela Gomes da Costa

DESAFIO:

 Desenvolver uma Análise Exploratória de Dados (AED) aplicada ao varejo para aprender como transformar dados brutos em informações úteis.

Link da Base de Dados: https://www.kaggle.com/datasets/namespaiva/base-varejo/data

- Sprint 1 (Importação dos dados)

In [2]:
# Importa o pandas para carregar, limpar e analisar os dados
%pip install pandas
import pandas as pd


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Para executar no VS Code, selecionar o kernel Python e usar **Executar Tudo**, com o CSV na mesma pasta do notebook. Caso o pandas não esteja instalado, executar `%pip install pandas` em uma célula separada, uma única vez.

In [3]:
# Carrega o arquivo CSV, que utiliza ponto e vírgula como separador
df_varejo = pd.read_csv("base_varejo.csv", sep=";")

# Exibe as primeiras cinco linhas da base
df_varejo.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


- Sprint 2 (Transformação de Strings, Integer e Float e Datetime)

In [4]:
# Mostra a quantidade de registros e de colunas
print("Número de registros:", df_varejo.shape[0])
print("Número de colunas:", df_varejo.shape[1])

# Mostra os nomes das colunas
print("\nColunas da base:")
print(df_varejo.columns.tolist())

# Mostra o tipo de dado de cada coluna
print("\nTipos de dados:")
print(df_varejo.dtypes)

Número de registros: 830000
Número de colunas: 14

Colunas da base:
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Tipos de dados:
DATA               str
CO_ID            int64
CL_ID            int64
CL_GENERO          str
CL_EC            int64
CL_FHL           int64
CL_SEG             str
PR_ID            int64
PR_CAT             str
PR_NOME            str
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object


Vemos que há 830000 registros e 14 colunas. Quatro delas possuem nomes como: Unnamed: 10, que já indicam um ponto de atenção que teremos que tratar no decorrer do desenvolvimento. Além disso, a coluna DATA foi carregada como texto, então será necessário convertê-la antes de analisar as compras ao longo do tempo.

Conforme o dicionário de dados disponibilizado no kaggle, CO_ID identifica a compra pelo número da nota fiscal, CL_ID identifica o cliente e PR_ID identifica o produto adquirido pelo seu SKU. Já PR_CAT informa a categoria do produto e PR_NOME, seu nome.

As características dos clientes estão nas colunas CL_GENERO, que representa o sexo biológico informado pelo cliente; CL_FHL, que informa o número de filhos; CL_SEG, que indica a classe econômica A, B ou C; e CL_EC, que representa o estado civil, com os códigos: 1 — casado ou união estável; 2 — divorciado; 3 — separado; 4 — solteiro; e 5 — viúvo.

Embora algumas dessas colunas contenham números, eles possuem funções diferentes. Os identificadores e os códigos de estado civil representam categorias, portanto calcular suas médias não teria significado para esta análise. O número de filhos, no entanto, é uma medida quantitativa e será utilizado nos cálculos estatísticos.

In [5]:
# Conta os valores ausentes em cada coluna
resumo_nulos = pd.DataFrame({
    "quantidade": df_varejo.isnull().sum(),
    "percentual": df_varejo.isnull().mean() * 100
})

print("Valores nulos por coluna:")
display(resumo_nulos)

Valores nulos por coluna:


,quantidade,percentual
DATA,0,0.0
CO_ID,0,0.0
CL_ID,0,0.0
CL_GENERO,0,0.0
CL_EC,0,0.0
CL_FHL,0,0.0
CL_SEG,0,0.0
PR_ID,0,0.0
PR_CAT,0,0.0
PR_NOME,0,0.0


O isnull() identificou valores ausentes nas quatro colunas extras. Nas demais colunas, não foram apontadas ausências.

No entanto, esse resultado não garante que todos os campos tenham informações válidas. Vou verificar os valores e suas frequências nas colunas de texto para identificar possíveis marcadores de informação ausente ou diferenças de preenchimento.

In [8]:
# Mostra os valores e suas frequências nas colunas de texto
colunas_texto = df_varejo.select_dtypes(
    include=["object", "string"]
).columns

for coluna in colunas_texto:
    print(f"\nColuna: {coluna}")
    print("Quantidade de valores distintos:", df_varejo[coluna].nunique())

    # Exibe todas as categorias encontradas, incluindo possíveis ausências
    display(
        df_varejo[coluna]
        .value_counts(dropna=False)
        .rename_axis("valor")
        .reset_index(name="quantidade")
    )


Coluna: DATA
Quantidade de valores distintos: 333


,valor,quantidade
0,21/08/2020,4641
1,15/04/2019,4628
2,23/10/2021,4569
3,22/01/2022,4539
4,31/01/2022,4539
...,...,...
328,19/01/2019,471
329,31/10/2019,468
330,31/10/2021,445
331,07/12/2021,433



Coluna: CL_GENERO
Quantidade de valores distintos: 2


,valor,quantidade
0,F,432576
1,M,397424



Coluna: CL_SEG
Quantidade de valores distintos: 3


,valor,quantidade
0,B,530163
1,C,232101
2,A,67736



Coluna: PR_CAT
Quantidade de valores distintos: 7


,valor,quantidade
0,ALIMENTOS,434767
1,HIGIENE,155574
2,LIMPEZA,145754
3,BEBIDAS,43299
4,PET,32399
5,ACESSORIOS,14557
6,#N/D,3650



Coluna: PR_NOME
Quantidade de valores distintos: 118


,valor,quantidade
0,PRESUNTO COZIDO,14381
1,SARDINHA,7490
2,GEL,7399
3,BANANA,7385
4,DESENGORDURANTE,7378
...,...,...
113,#N/D,3650
114,ABSORVENTE,3639
115,ALCOOL,3611
116,ALIMENTO PARA PASSARO,3605


Ao verificar os preenchimentos, encontrei o marcador #N/D nas colunas PR_CAT e PR_NOME. Embora esses campos não estejam vazios, o marcador não informa uma categoria ou um nome de produto válido.

Por isso, vou complementar a verificação de nulos com uma busca por marcadores textuais de informação ausente. Além do #N/D observado, verificarei possíveis variações: N/D, N/A e NULL. A inclusão dessas variações na busca não significa que elas estejam presentes na base.

In [9]:
# Verifica as datas sem modificar a coluna original
datas_convertidas = pd.to_datetime(
    df_varejo["DATA"], format="%d/%m/%Y", errors="coerce"
)
datas_invalidas = df_varejo["DATA"].notna() & datas_convertidas.isna()

print("Datas inválidas:", datas_invalidas.sum())
display(df_varejo.loc[datas_invalidas, ["DATA"]].head())

# Verifica categorias vazias e marcadores que representam informação ausente
categorias = df_varejo["PR_CAT"].astype("string").str.strip().str.upper()
categorias_vazias = categorias.isna() | categorias.fillna("").eq("")
categorias_nao_informadas = categorias.isin(["#N/D", "N/D", "N/A", "NULL"])

print("Categorias nulas, vazias ou somente com espaços:", categorias_vazias.sum())
print("Categorias com marcador de informação ausente:", categorias_nao_informadas.sum())
display(df_varejo.loc[categorias_nao_informadas, ["PR_ID", "PR_CAT", "PR_NOME"]].head())

Datas inválidas: 0


,DATA


Categorias nulas, vazias ou somente com espaços: 0
Categorias com marcador de informação ausente: 3650


,PR_ID,PR_CAT,PR_NOME
82,107,#N/D,#N/D
223,107,#N/D,#N/D
640,107,#N/D,#N/D
857,107,#N/D,#N/D
917,107,#N/D,#N/D


In [6]:
# Identifica linhas inteiramente iguais, contando as repetições após a primeira
duplicatas = df_varejo.duplicated()

print("Quantidade de registros duplicados:", duplicatas.sum())

# Exibe os registros identificados como repetidos
df_varejo[duplicatas]

Quantidade de registros duplicados: 96553


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
19,01/02/2019,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,NaN,NaN,NaN,NaN
40,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
46,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
49,01/02/2019,1000,534,M,4,1,C,69,BEBIDAS,REFRIGERANTE LIMaO,NaN,NaN,NaN,NaN
50,01/02/2019,1000,534,M,4,1,C,218,ALIMENTOS,BIFE DE COXAO MOLE,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
829989,19/08/2022,919822,155,F,2,0,B,34,ALIMENTOS,KETCHUP,NaN,NaN,NaN,NaN
829992,19/08/2022,919822,155,F,2,0,B,30,ALIMENTOS,FILE DE PEIXE,NaN,NaN,NaN,NaN
829995,19/08/2022,919822,155,F,2,0,B,183,ALIMENTOS,KETCHUP,NaN,NaN,NaN,NaN
829996,19/08/2022,919822,155,F,2,0,B,56,ALIMENTOS,QUEIJO MUSSARELA,NaN,NaN,NaN,NaN


Mostra-se acima quatro colunas totalmente nulas e 96.553 linhas repetidas. As quatro colunas extras aparecem porque o arquivo possui separadores adicionais ao final das linhas; como não possuem informação, vou removê-las.

Não encontrei datas inválidas nem categorias vazias. Porém, há 3.650 registros com #N/D na categoria e no nome do produto. Esse marcador não foi identificado como nulo na leitura inicial, por isso a verificação de texto complementa o isnull().

As linhas iguais serão tratadas como duplicatas para esta AED, mantendo a primeira ocorrência. Essa é uma hipótese de trabalho: sem quantidade ou identificador do item, não consigo confirmar se todas as repetições são erros ou se algumas representam múltiplas unidades do mesmo produto. Vou manter essa limitação explícita nas conclusões.